# Stage 1 + Stage 2 Test Runner

Run a query through flow routing (stage 1) and query understanding (stage 2) sequentially, with timing and full output for each stage.

In [ ]:
import sys, os, time, asyncio, json
from pathlib import Path

# Ensure project root is on the path so imports resolve.
project_root = str(Path.cwd().parent) if Path.cwd().name == "search_v2" else str(Path.cwd())
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from dotenv import load_dotenv
load_dotenv(Path(project_root) / ".env")

from implementation.llms.generic_methods import LLMProvider
from search_v2.stage_1 import route_query
from search_v2.stage_2 import understand_query

## Configuration

Set your query, LLM provider/model, and any extra kwargs below.

In [ ]:
# ---- Query ----
query = "all star wars movies in order"

# ---- Presets (uncomment one) ----
# Each preset sets provider, model, and kwargs for no-thinking mode.

# Kimi K2.5 — no thinking
# provider, model, kwargs = LLMProvider.KIMI, "kimi-k2.5", {"enable_thinking": False}

# GPT 5.4 Mini — no thinking
# provider, model, kwargs = LLMProvider.OPENAI, "gpt-5.4-mini", {"reasoning_effort": "none", "verbosity":"low"}

# Gemini 3 Flash — no thinking
provider, model, kwargs = LLMProvider.GEMINI, "gemini-3-flash-preview", {"thinking_config": {"thinking_budget": 0}}

# ---- Or set manually ----
# provider = LLMProvider.OPENAI
# model = "gpt-5-mini"
# kwargs = {"reasoning_effort": "low"}

print(f"Query:    {query}")
print(f"Provider: {provider.value}")
print(f"Model:    {model}")
print(f"Kwargs:   {kwargs or '(defaults)'}")

## Run Stage 1: Flow Routing

In [ ]:
start = time.perf_counter()
stage_1_response, stage_1_input_tokens, stage_1_output_tokens = await route_query(query)
stage_1_elapsed = time.perf_counter() - start

print(f"Stage 1 completed in {stage_1_elapsed:.2f}s")
print(f"Tokens — input: {stage_1_input_tokens:,}  output: {stage_1_output_tokens:,}\n")
print(stage_1_response.model_dump_json(indent=2))

## Run Stage 2: Query Understanding

Runs stage 2 for the stage-1 `primary_intent` plus any `alternative_intents` whose flow is **standard**. Exact-title and similarity intents skip this step (they don't need query decomposition).

In [ ]:
from schemas.enums import SearchFlow

stage_1_intents = [stage_1_response.primary_intent, *stage_1_response.alternative_intents]
stage_2_results = []  # list of (intent_index, intent_label, intent_rewrite, response, elapsed, in_tok, out_tok)

for i, intent in enumerate(stage_1_intents):
    if intent.flow != SearchFlow.STANDARD:
        print(f"Intent {i + 1} ({intent.flow.value}): skipping stage 2 — not a standard-flow query.\n")
        continue

    # Stage 2 receives the intent_rewrite from stage 1 as its query input.
    print(f"Intent {i + 1} ({intent.flow.value}): running stage 2 on intent_rewrite...")
    start = time.perf_counter()
    s2_response, s2_in, s2_out = await understand_query(
        query=intent.intent_rewrite, provider=provider, model=model, **kwargs
    )
    s2_elapsed = time.perf_counter() - start

    stage_2_results.append((i, intent.display_phrase, intent.intent_rewrite, s2_response, s2_elapsed, s2_in, s2_out))
    print(f"  Done in {s2_elapsed:.2f}s  (tokens — input: {s2_in:,}  output: {s2_out:,})\n")

if not stage_2_results:
    print("No standard-flow intents — stage 2 was not needed.")

## Summary & Full Results

In [ ]:
# ---- Timing summary ----
total_stage_2_time = sum(r[3] for r in stage_2_results)
total_time = stage_1_elapsed + total_stage_2_time

print("=" * 60)
print("TIMING SUMMARY")
print("=" * 60)
print(f"  Stage 1 (flow routing):      {stage_1_elapsed:.2f}s")
for idx, label, _, _, elapsed, _, _ in stage_2_results:
    print(f"  Stage 2 (intent {idx + 1}: {label}):  {elapsed:.2f}s")
print(f"  Total:                       {total_time:.2f}s")
print()

# ---- Token summary ----
total_in = stage_1_input_tokens + sum(r[5] for r in stage_2_results)
total_out = stage_1_output_tokens + sum(r[6] for r in stage_2_results)

print("TOKEN SUMMARY")
print("=" * 60)
print(f"  Stage 1:  {stage_1_input_tokens:>6,} in / {stage_1_output_tokens:>6,} out")
for idx, label, _, _, _, s2_in, s2_out in stage_2_results:
    print(f"  Stage 2 (intent {idx + 1}: {label}): {s2_in:>6,} in / {s2_out:>6,} out")
print(f"  Total:    {total_in:>6,} in / {total_out:>6,} out")
print()

# ---- Stage 1 full output ----
print("=" * 60)
print("STAGE 1 — FLOW ROUTING RESPONSE")
print("=" * 60)
print(stage_1_response.model_dump_json(indent=2))

# ---- Stage 2 full output per intent ----
for idx, label, intent, response, _, _, _ in stage_2_results:
    print()
    print("=" * 60)
    print(f"STAGE 2 — QUERY UNDERSTANDING (intent {idx + 1}: {label})")
    print(f"Input: {intent}")
    print("=" * 60)
    print(response.model_dump_json(indent=2))